# Model-conditioned inference of $D_{it}$ and $Q_f$ from the two PL ratios

This notebook asks a deliberately restricted inverse question:

> Within the coupled optical/transport/surface-SRH model, which combinations of
> $D_{it}$ and $Q_f$ reproduce the measured stack/bare PL ratios of **3.071 on
> n-InP** and **0.800 on p-InP**, and how does that inferred point move when
> uncertain experimental and model inputs are varied?

For every Monte Carlo draw, the notebook samples nuisance parameters and the two
measured PL ratios, then solves the two nonlinear equations

\[
\ln R_n(D_{it},Q_f;\theta)=\ln R_{n,\mathrm{draw}},\qquad
\ln R_p(D_{it},Q_f;\theta)=\ln R_{p,\mathrm{draw}},
\]

where $\theta$ denotes the sampled assumptions. The output cloud is therefore a
**model-conditioned uncertainty region**, not an independent electrical
measurement of $D_{it}$ or $Q_f$.

The defaults are intentionally transparent placeholders. Replace experimental
uncertainties with replicate/spot statistics when available. The expensive
production run is disabled by default; start with the validation run.


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.optimize import least_squares
from scipy.stats import qmc, spearmanr
from IPython.display import display

root = Path.cwd()
if not (root/'src').exists():
    root = root.parent
sys.path.insert(0, str(root/'src'))

import surpass
from surpass import (
    BulkModel, BulkTransportTable, InterfaceChargeModel, InterfaceDefectModel,
    Layer, OpticalStack, SteadyState1DSolver, SurfaceBoundaryTable,
    SurfaceSRHModel, Wafer, get_optical_material,
)

try:
    import scienceplots
    plt.style.use(['science', 'notebook', 'no-latex'])
except ImportError:
    plt.style.use('default')

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10})
output_dir = root/'examples'/'output'
output_dir.mkdir(parents=True, exist_ok=True)

print('surpass version:', surpass.__version__)
print('loaded from:', Path(surpass.__file__).resolve())


## 1. Fixed experimental geometry and nominal model

The bare reference has no optical coating, $Q_f=0$, and an uncertain
representative unpassivated $D_{it}$. Every candidate passivated point uses the
4-nm PO$_x$/13-nm AlO$_x$ pump optics. Thus optical and electrical passivation
are kept conceptually separate.


In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
rear_s_cm_s = 1e5

N_D_NOMINAL = 8.5e18
N_A_NOMINAL = 5.1e18
INTENSITY_N_NOMINAL = 1.0
INTENSITY_P_NOMINAL = 13.0

RATIO_N = 3.071
RATIO_P = 0.800

# Adamowicz et al., Vacuum 63, 223-227 (2001), DOI:
# 10.1016/S0042-207X(01)00195-6. These are literature-motivated
# starting values for chemically polished InP(100), not measured POx/InP values.
SIGMA_N_NOMINAL = 1e-14
SIGMA_P_NOMINAL = 1e-13
VTH_N = 4.13e7
VTH_P = 1.51e7

ENABLE_BGN = True
INCLUDE_INTERFACE_CHARGE = True
INTERFACE_CHARGE = InterfaceChargeModel(0.5, 0.02) if INCLUDE_INTERFACE_CHARGE else None

air = get_optical_material('air')
inp_optical = get_optical_material('InP_demo')
pox_optical = get_optical_material('POx_demo')
alox_optical = get_optical_material('AlOx_demo')
coated_pump = OpticalStack(
    air,
    [Layer(alox_optical, 13.0, 'AlOx'), Layer(pox_optical, 4.0, 'POx')],
    inp_optical,
)
bare_pump = OpticalStack(air, [], inp_optical)

# Slightly reduced mesh/table sizes make repeated inverse solutions practical.
# Increase these for the final convergence check.
edges_cm = np.r_[0.0, np.geomspace(1e-8, 5e-4, 80),
                 np.linspace(5e-4, thickness_cm, 66)[1:]]
TRANSPORT_POINTS = 121
BOUNDARY_POINTS = 61
ENERGY_POINTS = 101
MAX_NFEV_TRANSPORT = 900


## 2. Uncertainty model

The table below is the scientific heart of the exercise. The current ranges are
reasonable exploratory choices, not measured confidence intervals:

- positive quantities spanning decades are sampled in log space;
- doping and laser intensity use truncated normal-like ranges generated from a
  Latin-hypercube coordinate;
- the two experimental PL-ratio uncertainties are currently set to 5% relative;
- $D_{it,\mathrm{bare}}$ is sampled because the normalization point is assumed,
  rather than measured;
- capture cross sections are represented by an overall scale and a ratio. This
  separates recombination strength from electron/hole asymmetry.

Use replicate PL measurements to replace `ratio_rel_sigma` before treating the
cloud as a paper-quality interval.


In [ ]:
uncertainty = pd.DataFrame([
    ('PL ratio n-InP', 'normal', RATIO_N, 0.05, 'relative 1 sigma'),
    ('PL ratio p-InP', 'normal', RATIO_P, 0.05, 'relative 1 sigma'),
    ('bulk SRH lifetime', 'log-uniform', 2e-9, 10e-9, 's'),
    ('photon recycling probability', 'uniform', 0.0, 0.93, '-'),
    ('bare Dit', 'log-uniform', 3e11, 3e12, 'eV^-1 cm^-2'),
    ('capture cross-section scale', 'log-uniform', 10**-0.5, 10**0.5, 'x nominal'),
    ('sigma_p/sigma_n multiplier', 'log-uniform', 10**-0.5, 10**0.5, 'x nominal ratio'),
    ('n-InP donor density', 'uniform', 0.9, 1.1, 'x nominal'),
    ('p-InP acceptor density', 'uniform', 0.9, 1.1, 'x nominal'),
    ('n-InP intensity', 'uniform', 0.9, 1.1, 'x nominal'),
    ('p-InP intensity', 'uniform', 0.9, 1.1, 'x nominal'),
], columns=['quantity','distribution','low/mean','high/relative sigma','unit or note'])
display(uncertainty)

def log_uniform(u, low, high):
    return 10**(np.log10(low) + u*(np.log10(high)-np.log10(low)))

def make_draws(n, seed=20260929, ratio_rel_sigma=0.05):
    # Latin-hypercube sampling gives better coverage than plain random draws
    # for a modest number of expensive model evaluations.
    u = qmc.LatinHypercube(d=11, seed=seed).random(n)
    # Convert the two ratio coordinates to approximately normal quantiles,
    # clipping the extreme tails for stable small-sample designs.
    from scipy.stats import norm
    z_n = norm.ppf(np.clip(u[:,0], 0.005, 0.995))
    z_p = norm.ppf(np.clip(u[:,1], 0.005, 0.995))
    scale = log_uniform(u[:,5], 10**-0.5, 10**0.5)
    ratio_multiplier = log_uniform(u[:,6], 10**-0.5, 10**0.5)
    # Preserve the geometric mean scale while varying sigma_p/sigma_n.
    sigma_n = SIGMA_N_NOMINAL*scale/np.sqrt(ratio_multiplier)
    sigma_p = SIGMA_P_NOMINAL*scale*np.sqrt(ratio_multiplier)
    return pd.DataFrame({
        'ratio_n': RATIO_N*np.exp(ratio_rel_sigma*z_n),
        'ratio_p': RATIO_P*np.exp(ratio_rel_sigma*z_p),
        'tau_srh_s': log_uniform(u[:,2], 2e-9, 10e-9),
        'p_recycling': 0.93*u[:,3],
        'bare_dit': log_uniform(u[:,4], 3e11, 3e12),
        'sigma_n': sigma_n,
        'sigma_p': sigma_p,
        'donor_cm3': N_D_NOMINAL*(0.9 + 0.2*u[:,7]),
        'acceptor_cm3': N_A_NOMINAL*(0.9 + 0.2*u[:,8]),
        'intensity_n': INTENSITY_N_NOMINAL*(0.9 + 0.2*u[:,9]),
        'intensity_p': INTENSITY_P_NOMINAL*(0.9 + 0.2*u[:,10]),
    })

display(make_draws(8))


## 3. Forward model for one uncertainty draw

For computational efficiency, each draw constructs the bulk transport tables,
generation profiles, and bare references once. Only the surface boundary and
passivated transport solution are rebuilt during the two-parameter inverse fit.
The residual is expressed in logarithmic PL ratio, so fractional errors on the
two wafer types receive comparable weight.


In [ ]:
def _surface_model(wafer, charge_cm2, dit, sigma_n, sigma_p):
    defects = InterfaceDefectModel(
        dit, sigma_n, sigma_p, VTH_N, VTH_P,
        shape='constant', energy_points=ENERGY_POINTS,
    )
    return SurfaceSRHModel(
        BulkModel(wafer, enable_bgn=ENABLE_BGN), charge_cm2, defects,
        electrostatics='self_consistent', driving_force='excess_product',
        interface_charge=INTERFACE_CHARGE,
    )

class DrawForwardModel:
    def __init__(self, row):
        self.row = row
        self.labels = ('n-InP', 'p-InP')
        self.targets = np.array([row.ratio_n, row.ratio_p], dtype=float)
        self.parts = {}
        settings = {
            'n-InP': (Wafer('InP', donor_cm3=row.donor_cm3, thickness_um=thickness_um), row.intensity_n),
            'p-InP': (Wafer('InP', acceptor_cm3=row.acceptor_cm3, thickness_um=thickness_um), row.intensity_p),
        }
        for label, (wafer, intensity) in settings.items():
            bulk = BulkModel(wafer, enable_bgn=ENABLE_BGN, tau_srh_s=row.tau_srh_s)
            transport = BulkTransportTable.from_bulk_model(
                bulk, 1.0, 1e21, points=TRANSPORT_POINTS)
            solver = SteadyState1DSolver(transport)
            coated_generation = coated_pump.generation_cell_average_cm3_s(
                wavelength_nm, intensity, edges_cm)
            bare_generation = bare_pump.generation_cell_average_cm3_s(
                wavelength_nm, intensity, edges_cm)
            bare_surface = _surface_model(
                wafer, 0.0, row.bare_dit, row.sigma_n, row.sigma_p)
            bare_boundary = SurfaceBoundaryTable.from_surface_model(
                bare_surface, 1.0, 1e21, points=BOUNDARY_POINTS)
            bare_result = solver.solve(
                thickness_cm, bare_generation, bare_boundary, rear_s_cm_s,
                cell_edges_cm=edges_cm, max_nfev=MAX_NFEV_TRANSPORT,
                recycling_probability=row.p_recycling)
            if not bare_result.converged:
                raise RuntimeError(f'Bare reference did not converge for {label}')
            self.parts[label] = (wafer, solver, coated_generation,
                                 bare_result.internal_radiative_emission_cm2_s)
        self.cache = {}

    def ratios(self, log10_dit, q12):
        # Keep enough digits for the optimizer's finite-difference probes.
        # Coarse rounding here can make the perturbed point identical to the
        # current point, producing a zero numerical derivative and pinning a
        # parameter to its starting value.
        key = (round(float(log10_dit), 12), round(float(q12), 12))
        if key in self.cache:
            return self.cache[key]
        dit = 10**float(log10_dit)
        charge = float(q12)*1e12
        out = []
        for label in self.labels:
            wafer, solver, generation, bare_pl = self.parts[label]
            surface = _surface_model(
                wafer, charge, dit, self.row.sigma_n, self.row.sigma_p)
            boundary = SurfaceBoundaryTable.from_surface_model(
                surface, 1.0, 1e21, points=BOUNDARY_POINTS)
            result = solver.solve(
                thickness_cm, generation, boundary, rear_s_cm_s,
                cell_edges_cm=edges_cm, max_nfev=MAX_NFEV_TRANSPORT,
                recycling_probability=self.row.p_recycling)
            out.append(result.internal_radiative_emission_cm2_s/bare_pl
                       if result.converged else np.nan)
        self.cache[key] = np.asarray(out)
        return self.cache[key]

    def residual(self, x):
        pred = self.ratios(x[0], x[1])
        if np.any(~np.isfinite(pred)) or np.any(pred <= 0):
            return np.array([1e3, 1e3])
        return np.log(pred/self.targets)


## 4. Solve one nominal case and inspect the local likelihood surface

The bounded least-squares solution is started from several locations. This is
important because the p-InP response can have two branches on opposite sides of
the recombination valley. The likelihood plot exposes multiple or weakly
identified solutions that a single optimizer call could hide.


In [ ]:
LOGDIT_BOUNDS = (9.5, 12.8)
Q12_BOUNDS = (-1.0, 6.0)
STARTS = [(10.5, 1.0), (10.5, 2.5), (11.5, 4.0)]
EXACT_MATCH_TOL = 0.025  # RMS log-ratio error, about a few percent

def solve_draw(row, starts=STARTS):
    t0 = time.perf_counter()
    try:
        model = DrawForwardModel(row)
        candidates = []
        for x0 in starts:
            fit = least_squares(
                model.residual, x0=x0,
                bounds=([LOGDIT_BOUNDS[0], Q12_BOUNDS[0]],
                        [LOGDIT_BOUNDS[1], Q12_BOUNDS[1]]),
                xtol=2e-4, ftol=2e-4, gtol=2e-4, max_nfev=35,
                x_scale=[1.0, 2.0], diff_step=[2e-3, 2e-3],
            )
            candidates.append(fit)
        fit = min(candidates, key=lambda f: np.sum(f.fun**2))
        pred = model.ratios(*fit.x)
        at_boundary = (
            abs(fit.x[0]-LOGDIT_BOUNDS[0]) < 0.02 or
            abs(fit.x[0]-LOGDIT_BOUNDS[1]) < 0.02 or
            abs(fit.x[1]-Q12_BOUNDS[0]) < 0.03 or
            abs(fit.x[1]-Q12_BOUNDS[1]) < 0.03
        )
        return {
            'success': bool(fit.success and np.all(np.isfinite(pred))),
            'log10_dit_fit': fit.x[0], 'dit_fit': 10**fit.x[0],
            'qf_over_q_fit_cm2': fit.x[1]*1e12,
            'pred_ratio_n': pred[0], 'pred_ratio_p': pred[1],
            'rms_log_residual': np.sqrt(np.mean(model.residual(fit.x)**2)),
            'at_boundary': at_boundary, 'n_forward_points': len(model.cache),
            'seconds': time.perf_counter()-t0, 'message': fit.message,
        }, model
    except Exception as exc:
        return {
            'success': False, 'log10_dit_fit': np.nan, 'dit_fit': np.nan,
            'qf_over_q_fit_cm2': np.nan, 'pred_ratio_n': np.nan,
            'pred_ratio_p': np.nan, 'rms_log_residual': np.inf,
            'at_boundary': False, 'n_forward_points': 0,
            'seconds': time.perf_counter()-t0, 'message': repr(exc),
        }, None

nominal = make_draws(1, seed=1).iloc[0].copy()
nominal[['ratio_n','ratio_p','tau_srh_s','p_recycling','bare_dit',
         'sigma_n','sigma_p','donor_cm3','acceptor_cm3','intensity_n','intensity_p']] = [
    RATIO_N, RATIO_P, 5e-9, 0.93, 1e12,
    SIGMA_N_NOMINAL, SIGMA_P_NOMINAL,
    N_D_NOMINAL, N_A_NOMINAL, INTENSITY_N_NOMINAL, INTENSITY_P_NOMINAL,
]
nominal_result, nominal_model = solve_draw(nominal)
display(pd.DataFrame([nominal_result]))
if nominal_result['rms_log_residual'] > EXACT_MATCH_TOL:
    print('WARNING: the tested nominal model has no exact common intersection '
          'within the selected branches/bounds. Enable the diagnostic grid to '
          'verify the global landscape before interpreting an inferred point.')


In [ ]:
RUN_DIAGNOSTIC_GRID = False  # expensive; enable for the production SI run
if RUN_DIAGNOSTIC_GRID:
    # This grid is not used by the Monte Carlo fit; it reveals branch structure.
    logdit_axis = np.linspace(9.7, 12.5, 18)
    q12_axis = np.linspace(-0.5, 5.5, 20)
    misfit = np.empty((logdit_axis.size, q12_axis.size))
    for iy, ld in enumerate(logdit_axis):
        for ix, q12 in enumerate(q12_axis):
            r = nominal_model.residual((ld, q12))
            misfit[iy, ix] = np.sqrt(np.mean(r*r))

    fig, ax = plt.subplots(figsize=(7.2, 4.7), constrained_layout=True)
    cf = ax.contourf(q12_axis, 10**logdit_axis, misfit,
                     levels=np.geomspace(max(1e-3, np.nanmin(misfit)), 1.0, 24),
                     norm=LogNorm(), cmap='viridis_r', extend='max')
    ax.plot(nominal_result['qf_over_q_fit_cm2']/1e12,
            nominal_result['dit_fit'], 'o', color='red', mec='white', mew=0.8,
            label='best nominal match')
    ax.set(yscale='log', xlabel=r'$Q_f/q$ ($10^{12}$ cm$^{-2}$)',
           ylabel=r'$D_{it}$ (eV$^{-1}$ cm$^{-2}$)',
           title='Nominal joint misfit in the two PL ratios')
    ax.legend(fontsize=8)
    fig.colorbar(cf, ax=ax, label='RMS logarithmic ratio residual')
    fig.savefig(output_dir/'paper_inp_si_nominal_inverse_landscape.png', bbox_inches='tight')
else:
    print('Diagnostic grid skipped; set RUN_DIAGNOSTIC_GRID=True for the SI run.')


## 5. Monte Carlo inverse propagation

The default validation run is deliberately small. For the SI, increase
`N_DRAWS` to roughly 200--500 after checking runtime and numerical convergence.
Latin-hypercube sampling makes 200 well-spread draws more useful than 200 plain
random draws. Save the resulting table so a long run does not need to be
repeated merely to change a plot.

A solution is considered usable only when it converges, is not at a search
boundary, and reproduces both ratios to the selected logarithmic tolerance.
The fraction rejected is itself a result and should be reported.


In [ ]:
RUN_MONTE_CARLO = True
N_DRAWS = 4        # smoke test; use 200-500 for the production SI run
SEED = 20260929
RATIO_REL_SIGMA = 0.05  # replace by experimental replicate uncertainty
MAX_RMS_LOG_RESIDUAL = EXACT_MATCH_TOL

draws = make_draws(N_DRAWS, seed=SEED, ratio_rel_sigma=RATIO_REL_SIGMA)
records = []
if RUN_MONTE_CARLO:
    for i, row in draws.iterrows():
        result, _ = solve_draw(row)
        records.append({**row.to_dict(), **result})
        print(f'{i+1:4d}/{len(draws)}  success={result["success"]}  '
              f'Dit={result["dit_fit"]:.3g}  '
              f'Qf/q={result["qf_over_q_fit_cm2"]:.3g}  '
              f'rms={result["rms_log_residual"]:.3g}  '
              f'{result["seconds"]:.1f} s')
    mc = pd.DataFrame(records)
    mc['usable_best_fit'] = mc.success & ~mc.at_boundary
    mc['compatible'] = (mc.usable_best_fit &
                        (mc.rms_log_residual <= MAX_RMS_LOG_RESIDUAL))
    csv_path = output_dir/'inp_dit_qf_monte_carlo_results.csv'
    mc.to_csv(csv_path, index=False)
    print('saved:', csv_path)
else:
    csv_path = output_dir/'inp_dit_qf_monte_carlo_results.csv'
    mc = pd.read_csv(csv_path)

display(mc[['success','usable_best_fit','compatible','at_boundary','dit_fit','qf_over_q_fit_cm2',
            'pred_ratio_n','pred_ratio_p','rms_log_residual','seconds']])
print(f'Exactly compatible: {mc.compatible.sum()} of {len(mc)} draws '
      f'({100*mc.compatible.mean():.1f}%).')


## 6. Inferred cloud and marginal intervals

The scatter plot is preferable to drawing a single ellipse when the inverse
problem is curved, multimodal, or truncated by the model bounds. The marginal
percentiles summarize the cloud but should not be called confidence intervals
unless the input distributions have been justified statistically.


In [ ]:
ok = mc.loc[mc.compatible].copy()
best = mc.loc[mc.usable_best_fit].copy()
if len(ok) < 3:
    print('Too few exactly compatible draws for an uncertainty interval. This means '
          'that the model assumptions generally do not allow the two measured-ratio '
          'contours to intersect within the selected tolerance. The plot below shows '
          'best-compromise points only; do not report them as extracted parameters.')
    plot_data = best
else:
    plot_data = ok
    summary = pd.DataFrame({
        'quantity': [r'$D_{it}$ (eV$^{-1}$ cm$^{-2}$)', r'$Q_f/q$ (cm$^{-2}$)'],
        'p16': [ok.dit_fit.quantile(.16), ok.qf_over_q_fit_cm2.quantile(.16)],
        'median': [ok.dit_fit.median(), ok.qf_over_q_fit_cm2.median()],
        'p84': [ok.dit_fit.quantile(.84), ok.qf_over_q_fit_cm2.quantile(.84)],
        'p2.5': [ok.dit_fit.quantile(.025), ok.qf_over_q_fit_cm2.quantile(.025)],
        'p97.5': [ok.dit_fit.quantile(.975), ok.qf_over_q_fit_cm2.quantile(.975)],
    })
    display(summary)

    fig = plt.figure(figsize=(9.0, 7.2), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, width_ratios=(4, 1.2), height_ratios=(1.2, 4))
    ax_histx = fig.add_subplot(gs[0,0])
    ax = fig.add_subplot(gs[1,0], sharex=ax_histx)
    ax_histy = fig.add_subplot(gs[1,1], sharey=ax)
    color = plot_data.rms_log_residual
    sc = ax.scatter(plot_data.qf_over_q_fit_cm2/1e12, plot_data.dit_fit, c=color,
                    cmap='magma_r', s=34, alpha=.85, edgecolor='white', linewidth=.35)
    ax.plot(nominal_result['qf_over_q_fit_cm2']/1e12,
            nominal_result['dit_fit'], '*', color='red', mec='black', ms=12,
            label='nominal model')
    ax.set(yscale='log', xlabel=r'Matched $Q_f/q$ ($10^{12}$ cm$^{-2}$)',
           ylabel=r'Matched $D_{it}$ (eV$^{-1}$ cm$^{-2}$)')
    ax.legend(fontsize=8)
    ax_histx.hist(plot_data.qf_over_q_fit_cm2/1e12, bins='auto', color='.45')
    dit_bins = np.geomspace(plot_data.dit_fit.min()*.95, plot_data.dit_fit.max()*1.05,
                            max(4, int(np.sqrt(len(plot_data)))+1))
    ax_histy.hist(plot_data.dit_fit, bins=dit_bins, orientation='horizontal', color='.45')
    ax_histy.set_yscale('log')
    ax_histy.tick_params(labelleft=False)
    ax_histx.tick_params(labelbottom=False)
    fig.colorbar(sc, ax=[ax,ax_histx], label='best-fit RMS logarithmic residual')
    fig.suptitle('Model-conditioned compatible region' if len(ok) >= 3
                 else 'Best compromises: no robust common contour intersection')
    fig.savefig(output_dir/'paper_inp_si_dit_qf_uncertainty_cloud.png', bbox_inches='tight')


## 7. Which assumptions move the inferred point?

Rank correlations are a screening tool, not a causal decomposition. They reveal
which assumptions deserve dedicated one-at-a-time plots and stronger literature
or experimental constraints. For a paper-quality global sensitivity analysis,
increase the sample size and consider Sobol indices after constructing a
validated surrogate model.


In [ ]:
if len(ok) >= 8:
    inputs = ['ratio_n','ratio_p','tau_srh_s','p_recycling','bare_dit',
              'sigma_n','sigma_p','donor_cm3','acceptor_cm3',
              'intensity_n','intensity_p']
    outputs = ['log10_dit_fit','qf_over_q_fit_cm2']
    rows = []
    for inp in inputs:
        for out in outputs:
            rho, pvalue = spearmanr(ok[inp], ok[out])
            rows.append({'input':inp,'output':out,'Spearman rho':rho,'p':pvalue})
    sensitivity = pd.DataFrame(rows)
    display(sensitivity.pivot(index='input',columns='output',values='Spearman rho'))

    pivot = sensitivity.pivot(index='input',columns='output',values='Spearman rho')
    fig, ax = plt.subplots(figsize=(7.5,5.0), constrained_layout=True)
    im = ax.imshow(pivot.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=20, ha='right')
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f'{pivot.iloc[i,j]:+.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, label='Spearman rank correlation')
    ax.set_title('Screening sensitivity of matched parameters')
    fig.savefig(output_dir/'paper_inp_si_inverse_sensitivity.png', bbox_inches='tight')
else:
    print('Use at least ~50 exactly compatible samples before interpreting rank sensitivities.')


## 8. Diagnostics and reporting checklist

Before using the cloud in the supplementary information:

1. Replace the placeholder 5% PL-ratio uncertainty with replicate, spot-to-spot,
   or sample-to-sample statistics.
2. Run at least 200--500 Latin-hypercube draws and report the random seed,
   distributions, parameter ranges, rejection fraction, and solver bounds.
3. Repeat a subset with a finer spatial mesh, more transport/boundary points,
   and tighter optimizer tolerances to demonstrate numerical convergence.
4. Inspect failed and boundary-hitting draws. A large fraction indicates that
   the assumed model family cannot reproduce the observations or that a second
   solution branch lies outside the chosen bounds.
5. Plot the nominal likelihood surface and several extreme draws to check for
   multiple intersections. If two branches occur, report both rather than
   forcing the optimizer onto the branch near $Q_f/q\approx2\times10^{12}$ cm$^{-2}$.
6. Describe the result as a model-conditioned compatible region. The robust
   conclusions are likely the sign and approximate scale of $Q_f$; the exact
   numerical value of $D_{it}$ is more dependent on capture parameters and the
   assumed unpassivated reference.

An informative SI figure set would contain: (a) the nominal joint-misfit map,
(b) the Monte Carlo $(D_{it},Q_f)$ cloud with marginals, and (c) a compact
sensitivity ranking. The full sampled table should be included with the shared
code/data archive.


In [ ]:
# Compact failure audit
audit = pd.DataFrame({
    'category': ['exactly compatible','optimizer/model failure','boundary solution',
                 'residual above threshold'],
    'count': [mc.compatible.sum(), (~mc.success).sum(),
              (mc.success & mc.at_boundary).sum(),
              (mc.success & ~mc.at_boundary &
               (mc.rms_log_residual > MAX_RMS_LOG_RESIDUAL)).sum()],
})
display(audit)

if (~mc.compatible).any():
    display(mc.loc[~mc.compatible, [
        'success','at_boundary','rms_log_residual','message',
        'ratio_n','ratio_p','tau_srh_s','p_recycling','bare_dit',
        'sigma_n','sigma_p'
    ]])
